# Travel Reimbursement Approval Agent

An agent that reads a travel claim, pulls the relevant policy rules, runs a handful of checking tools, and returns a structured decision (`APPROVE`, `PARTIAL_APPROVE`, `REJECT` or `MANUAL_REVIEW`). It is built against the mock policy (Appendix A) and the five sample claims (Appendix B) from the assignment.

## README

**Setup**

```bash
pip install pandas matplotlib            # required
pip install ipywidgets openai            # optional 
```

**Environment variables (all optional)**

| Variable | Purpose |
|---|---|
| `OPENAI_API_KEY` | If set (and `openai` is installed) the LLM drives the tool calls and writes the explanation. |
| `OPENAI_MODEL` | Model name override. Defaults to `gpt-4o`. |

**Flow**

```
claim JSON -> lookup_policy -> check_eligibility -> (if anything is eligible)
           -> check_receipts -> check_limits -> check_timeliness -> check_approval_tier
           -> decision rules -> validate_output -> result JSON
```

In [ ]:
import os, re, json
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
API_KEY = os.getenv("OPENAI_API_KEY")

client = None
if API_KEY:
    try:
        import openai
        client = openai.OpenAI(api_key=API_KEY)
    except ImportError:
        print("OPENAI_API_KEY is set but the 'openai' package is missing - falling back to offline mode.")

MODE = "llm" if client else "offline"
print(f"Running in {MODE} mode" + (f" (model: {MODEL})" if client else ""))

## 1. Policy and limits

We store the mock policy rules (Appendix A) in a dictionary keyed by rule IDs so the agent can easily cite specific `POL-*` tags. All numeric thresholds live in a single `LIMITS` dictionary to keep our checking tools clean.

In [ ]:
POLICY = {
    "POL-CAT-01": {"title": "Eligible categories",
        "text": "Airfare (economy only), lodging, meals (per-diem), ground transport, and conference/registration fees are reimbursable when incurred for a documented business purpose.",
        "tags": ["eligible", "category", "airfare", "lodging", "meals", "ground", "transport", "conference", "registration"]},
    "POL-CAT-02": {"title": "Ineligible items",
        "text": "Alcohol, minibar, spa, gym, personal entertainment, in-room movies, shopping, gifts, fines/penalties/late fees and any personal expense are never reimbursable and are deducted in full.",
        "tags": ["ineligible", "alcohol", "minibar", "spa", "gym", "entertainment", "movies", "shopping", "gifts", "fines", "personal"]},
    "POL-PD-01": {"title": "Meals per-diem",
        "text": "Meals are capped at $75 per day. Amounts above the daily cap are deducted.",
        "tags": ["meals", "meal", "per-diem", "perdiem", "daily", "cap", "dinner"]},
    "POL-PD-02": {"title": "Lodging cap",
        "text": "Lodging is capped at $200 per night. Amounts above the nightly cap are deducted.",
        "tags": ["lodging", "hotel", "night", "nightly", "cap"]},
    "POL-PD-03": {"title": "Ground transport cap",
        "text": "Ground transport is capped at $50 per day. Amounts above the cap are deducted.",
        "tags": ["ground", "transport", "taxi", "rideshare", "train", "rental", "parking", "daily", "cap"]},
    "POL-AIR-01": {"title": "Airfare class",
        "text": "Only economy airfare is reimbursable. Business/first class is a policy exception and goes to Manual Review (a pre-approval may exist), not auto-deducted.",
        "tags": ["airfare", "flight", "class", "economy", "business", "first"]},
    "POL-RCT-01": {"title": "Receipt required above $25",
        "text": "Any single line item over $25 needs an itemized receipt. Airfare and lodging always need one regardless of amount.",
        "tags": ["receipt", "itemized", "airfare", "lodging", "attachment"]},
    "POL-RCT-02": {"title": "Missing receipt handling",
        "text": "A missing required receipt is not silently rejected; the claim is routed to Manual Review so the reviewer can request it.",
        "tags": ["receipt", "missing", "manual"]},
    "POL-APR-01": {"title": "Auto-approve tier",
        "text": "Total reimbursable amount <= $500 can be auto-approved if fully compliant.",
        "tags": ["approval", "threshold", "tier", "auto", "total"]},
    "POL-APR-02": {"title": "Manager tier",
        "text": "Total > $500 and <= $2,000 is approvable when fully compliant.",
        "tags": ["approval", "threshold", "tier", "manager", "total"]},
    "POL-APR-03": {"title": "Director / manual-review tier",
        "text": "Total > $2,000 exceeds the agent's authority and must go to Manual Review (director approval), even if otherwise compliant.",
        "tags": ["approval", "threshold", "tier", "director", "total", "high", "value"]},
    "POL-TIME-01": {"title": "Submission window",
        "text": "Claims must be submitted within 30 days of the expense date. Late claims go to Manual Review.",
        "tags": ["late", "timeliness", "submission", "window", "days", "deadline"]},
}

LIMITS = {
    "meals_per_day": 75,
    "lodging_per_night": 200,
    "ground_per_day": 50,
    "receipt_threshold": 25,
    "always_receipt": {"airfare", "lodging"},
    "auto_tier_max": 500,
    "manager_tier_max": 2000,
    "submission_window_days": 30,
}

ELIGIBLE = {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
INELIGIBLE = {"alcohol", "minibar", "spa", "gym", "entertainment", "movies", "shopping", "gifts", "fines", "personal"}

DECISIONS = ["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
OUTPUT_KEYS = ["claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs",
               "policy_refs", "confidence", "explanation", "tools_used"]

## 2. Claim intake

The five claims from Appendix B, entered as JSON-style dicts. A few quantities that the appendix only gives in the description text (number of nights/days, fare class, party size) are lifted into explicit fields so the tools don't have to parse free text. Nothing was added beyond what the appendix states.

In [ ]:
CLAIMS_RAW = [
    {"claim_id": "CLM-001", "employee": "A. Rivera", "purpose": "Attend 2-day industry conference (business)",
     "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20", "stated_total": 1110.00,
     "items": [
         {"category": "airfare", "description": "Round-trip economy airfare", "amount": 420.00, "receipt": True, "fare_class": "economy"},
         {"category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": 360.00, "receipt": True, "nights": 2},
         {"category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": 180.00, "receipt": True, "days": 3},
         {"category": "conference_fees", "description": "Conference registration", "amount": 150.00, "receipt": True},
     ]},
    {"claim_id": "CLM-002", "employee": "B. Osei", "purpose": "Weekend hotel stay",
     "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25", "stated_total": 380.00,
     "items": [
         {"category": "spa", "description": "Hotel spa package", "amount": 300.00, "receipt": True},
         {"category": "minibar", "description": "In-room minibar", "amount": 80.00, "receipt": True},
     ]},
    {"claim_id": "CLM-003", "employee": "C. Nakamura", "purpose": "Client site visit (business)",
     "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22", "stated_total": 940.00,
     "items": [
         {"category": "airfare", "description": "Round-trip economy airfare", "amount": 300.00, "receipt": True, "fare_class": "economy"},
         {"category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": 500.00, "receipt": True, "nights": 2},
         {"category": "meals", "description": "Meals, 2 days @ $70/day", "amount": 140.00, "receipt": True, "days": 2},
     ]},
    {"claim_id": "CLM-004", "employee": "D. Fischer", "purpose": "International vendor negotiation (business)",
     "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28", "stated_total": 3000.00,
     "items": [
         {"category": "airfare", "description": "Business-class international airfare", "amount": 2400.00, "receipt": True, "fare_class": "business"},
         {"category": "lodging", "description": "Hotel, 3 nights", "amount": 600.00, "receipt": False, "nights": 3},
     ]},
    {"claim_id": "CLM-005", "employee": "E. Haddad", "purpose": "Client dinner / business development",
     "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24", "stated_total": 220.00,
     "items": [
         {"category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.00, "receipt": False, "days": 1, "attendees": 4},
     ]},
]
CLAIMS = {c["claim_id"]: c for c in CLAIMS_RAW}

def money(x):
    return f"${x:,.2f}"

pd.DataFrame([{"claim_id": c["claim_id"], "employee": c["employee"], "items": len(c["items"]),
               "claimed": sum(i["amount"] for i in c["items"])} for c in CLAIMS_RAW])

## 3. Tools

Seven small functions. Each takes a `claim_id` (or a query / amount) and returns a plain dict so the output can be logged in the audit trail and handed back to an LLM as a tool result.


In [ ]:
def _tokens(text):
    return set(re.findall(r"[a-z][a-z\-]*", text.lower()))

def lookup_policy(query, top_k=4):
    """Rank policy rules by keyword overlap. Tag hits count extra."""
    q = _tokens(query)
    scored = []
    for pid, rule in POLICY.items():
        body = _tokens(rule["title"] + " " + rule["text"])
        tags = set(rule["tags"])
        score = len(q & body) + 3 * len(q & tags)
        if score:
            scored.append((score, pid))
    scored.sort(reverse=True)
    return {"query": query,
            "matches": [{"id": pid, "title": POLICY[pid]["title"], "text": POLICY[pid]["text"]}
                        for _, pid in scored[:top_k]]}


def check_eligibility(claim_id):
    c = CLAIMS[claim_id]
    items, exceptions, conflicts = [], [], []
    for idx, it in enumerate(c["items"]):
        cat = it["category"]
        if cat in ELIGIBLE:
            status, ref = "eligible", "POL-CAT-01"
        elif cat in INELIGIBLE:
            status, ref = "ineligible", "POL-CAT-02"
        else:
            status, ref = "unknown", None
            conflicts.append({"code": "UNKNOWN_CATEGORY", "ref": None,
                              "msg": f"category '{cat}' ({it['description']}) is not covered by the policy"})
        if cat == "airfare" and it.get("fare_class", "economy") != "economy":
            exceptions.append({"code": "AIRFARE_CLASS", "ref": "POL-AIR-01",
                               "msg": f"{it['fare_class']}-class airfare ({money(it['amount'])}) is a policy exception that may have a pre-approval (POL-AIR-01)"})
        items.append({"idx": idx, "category": cat, "description": it["description"],
                      "amount": it["amount"], "status": status, "ref": ref})

    summed = round(sum(i["amount"] for i in c["items"]), 2)
    if abs(summed - c["stated_total"]) > 0.005:
        conflicts.append({"code": "TOTAL_MISMATCH", "ref": None,
                          "msg": f"line items add up to {money(summed)} but the claim states {money(c['stated_total'])}"})
    return {"items": items, "exceptions": exceptions, "conflicts": conflicts,
            "eligible_count": sum(i["status"] == "eligible" for i in items),
            "ineligible_count": sum(i["status"] == "ineligible" for i in items)}


def check_receipts(claim_id):
    c = CLAIMS[claim_id]
    missing, checked = [], []
    for idx, it in enumerate(c["items"]):
        if it["category"] not in ELIGIBLE:
            continue  
        needed = it["amount"] > LIMITS["receipt_threshold"] or it["category"] in LIMITS["always_receipt"]
        checked.append({"idx": idx, "category": it["category"], "receipt_required": needed, "receipt_attached": it["receipt"]})
        if needed and not it["receipt"]:
            missing.append({"idx": idx, "category": it["category"],
                            "doc": f"{it['category']} receipt ({it['description']}, {money(it['amount'])})"})
    return {"checked": checked, "missing": missing,
            "refs": ["POL-RCT-01", "POL-RCT-02"] if missing else ["POL-RCT-01"]}


def check_limits(claim_id):
    """Apply per-diem / nightly caps to eligible items. Items with unclear units are flagged, not guessed."""
    c = CLAIMS[claim_id]
    lines, ambiguities = [], []
    for idx, it in enumerate(c["items"]):
        cat, amt = it["category"], it["amount"]
        if cat not in ELIGIBLE:
            continue
        cap, ref, unit = None, None, None
        if cat == "meals":
            ref, unit = "POL-PD-01", "day"
            days = it.get("days")
            if not days:
                ambiguities.append({"idx": idx, "ref": ref, "msg": "meals claimed without a day count, cannot apply the daily cap"})
            elif it.get("attendees", 1) > 1:
                ambiguities.append({"idx": idx, "ref": ref,
                                    "msg": f"group meal for {it['attendees']} people ({money(amt)}): the policy doesn't say whether the $75/day cap is per person or per claim"})
            else:
                cap = LIMITS["meals_per_day"] * days
        elif cat == "lodging":
            ref, unit = "POL-PD-02", "night"
            nights = it.get("nights")
            if not nights:
                ambiguities.append({"idx": idx, "ref": ref, "msg": "lodging claimed without a night count, cannot apply the nightly cap"})
            else:
                cap = LIMITS["lodging_per_night"] * nights
        elif cat == "ground_transport":
            ref, unit = "POL-PD-03", "day"
            days = it.get("days")
            if not days:
                ambiguities.append({"idx": idx, "ref": ref, "msg": "ground transport claimed without a day count, cannot apply the daily cap"})
            else:
                cap = LIMITS["ground_per_day"] * days
        allowed = amt if cap is None else min(amt, cap)
        lines.append({"idx": idx, "category": cat, "claimed": amt, "cap": cap, "allowed": round(allowed, 2),
                      "deducted": round(amt - allowed, 2), "ref": ref if amt > (cap if cap is not None else amt) else None})
    return {"lines": lines, "ambiguities": ambiguities,
            "reimbursable_total": round(sum(l["allowed"] for l in lines), 2),
            "cap_deductions": round(sum(l["deducted"] for l in lines), 2)}


def check_timeliness(claim_id):
    c = CLAIMS[claim_id]
    gap = (date.fromisoformat(c["submitted"]) - date.fromisoformat(c["trip_end"])).days
    return {"days_since_expense": gap, "late": gap > LIMITS["submission_window_days"], "ref": "POL-TIME-01"}


def check_approval_tier(reimbursable_total):
    t = reimbursable_total
    if t <= LIMITS["auto_tier_max"]:
        return {"total": t, "tier": "auto", "ref": "POL-APR-01", "needs_review": False}
    if t <= LIMITS["manager_tier_max"]:
        return {"total": t, "tier": "manager", "ref": "POL-APR-02", "needs_review": False}
    return {"total": t, "tier": "director", "ref": "POL-APR-03", "needs_review": True}


def validate_output(result, claimed_total):
    """Schema + consistency check run on every result before it is returned."""
    errs = []
    if set(result) != set(OUTPUT_KEYS):
        errs.append(f"keys differ from spec: {sorted(set(result) ^ set(OUTPUT_KEYS))}")
    if result.get("decision") not in DECISIONS:
        errs.append(f"invalid decision {result.get('decision')!r}")
    for k in ("approved_amount", "deducted_amount"):
        v = result.get(k)
        if not isinstance(v, (int, float)) or v < 0:
            errs.append(f"{k} must be a non-negative number")
    if isinstance(result.get("approved_amount"), (int, float)) and isinstance(result.get("deducted_amount"), (int, float)):
        if result["approved_amount"] + result["deducted_amount"] > claimed_total + 0.005:
            errs.append("approved + deducted exceeds the claimed total")
    conf = result.get("confidence")
    if not isinstance(conf, (int, float)) or not 0 <= conf <= 1:
        errs.append("confidence must be between 0 and 1")
    unknown = [r for r in result.get("policy_refs", []) if r not in POLICY]
    if unknown:
        errs.append(f"unknown policy refs: {unknown}")
    d = result.get("decision")
    if d == "APPROVE" and (result.get("deducted_amount") or result.get("missing_docs")):
        errs.append("APPROVE with deductions or missing docs")
    if d == "REJECT" and result.get("approved_amount"):
        errs.append("REJECT with an approved amount")
    if d == "MANUAL_REVIEW" and not result.get("explanation"):
        errs.append("MANUAL_REVIEW needs an explanation for the reviewer")
    return {"valid": not errs, "errors": errs}

## 4. Agent workflow

Two layers, kept deliberately separate:

1. **Exploration** - something decides which tools to call. In `llm` mode that is the model (it sees the claim, picks tools, reads the results, and finally calls `submit_decision`). In `offline` mode a small planner does it, and it still branches: if nothing on the claim is eligible it stops after the eligibility check instead of running receipts/limits/tier checks.
2. **Decision** - a deterministic function (`decide`) turns the tool findings into the decision, amounts, refs and missing docs. Money and policy logic is not left to the model.

If the LLM skipped a check that was needed, the harness runs it. If the LLM's decision disagrees with `decide`, the rule-based decision wins and the disagreement is recorded in the audit trail. The LLM contributes the tool-use flow and the written explanation, not the numbers.

In [ ]:
TOOL_FUNCS = {
    "lookup_policy": lambda cid, a: lookup_policy(a.get("query", "")),
    "check_eligibility": lambda cid, a: check_eligibility(cid),
    "check_receipts": lambda cid, a: check_receipts(cid),
    "check_limits": lambda cid, a: check_limits(cid),
    "check_timeliness": lambda cid, a: check_timeliness(cid),
    "check_approval_tier": lambda cid, a: check_approval_tier(float(a["reimbursable_total"])),
}

AUDIT = {}   


def call_tool(cid, name, args, findings, caller):
    out = TOOL_FUNCS[name](cid, args)
    AUDIT[cid].append({"caller": caller, "tool": name, "args": args, "output": out})
    findings[name] = out
    return out


def complete_findings(cid, findings):
    """Run any check the explorer didn't, so decide() always has what it needs."""
    def have(n): return n in findings

    if not have("lookup_policy"):
        cats = " ".join(sorted({i["category"] for i in CLAIMS[cid]["items"]}))
        call_tool(cid, "lookup_policy", {"query": f"{cats} receipt limit approval submission"}, findings, "harness")
    if not have("check_eligibility"):
        call_tool(cid, "check_eligibility", {}, findings, "harness")

    elig = findings["check_eligibility"]
    if elig["eligible_count"] == 0 and not any(i["status"] == "unknown" for i in elig["items"]):
        return findings  

    for name in ("check_receipts", "check_limits", "check_timeliness"):
        if not have(name):
            call_tool(cid, name, {}, findings, "harness")

    total = findings["check_limits"]["reimbursable_total"]
    tier = findings.get("check_approval_tier")
    if tier is None or abs(tier["total"] - total) > 0.005:   
        call_tool(cid, "check_approval_tier", {"reimbursable_total": total}, findings, "harness")
    return findings


def decide(cid, f):
    """Turn tool findings into a decision. Pure rules, no LLM."""
    c = CLAIMS[cid]
    claimed = round(sum(i["amount"] for i in c["items"]), 2)
    elig = f["check_eligibility"]
    rec, lim = f.get("check_receipts"), f.get("check_limits")
    tim, tier = f.get("check_timeliness"), f.get("check_approval_tier")

    refs, manual, notes = set(), [], []   

    ineligible = [i for i in elig["items"] if i["status"] == "ineligible"]
    ineligible_total = round(sum(i["amount"] for i in ineligible), 2)
    if ineligible:
        refs.add("POL-CAT-02")
        names = " and ".join(f"{i['description']} ({money(i['amount'])})" for i in ineligible)
        notes.append(f"{names} {'is' if len(ineligible) == 1 else 'are'} ineligible under POL-CAT-02")
    if elig["eligible_count"]:
        refs.add("POL-CAT-01")

    for e in elig["exceptions"]:
        manual.append(("policy", e["ref"], e["msg"]))
    for cf in elig["conflicts"]:
        manual.append(("ambiguity", cf["ref"], cf["msg"]))

    missing_docs = []
    if rec:
        refs.add("POL-RCT-01")
        for m in rec["missing"]:
            missing_docs.append(m["doc"])
            manual.append(("policy", "POL-RCT-02", f"the required {m['doc']} is missing (POL-RCT-01 / POL-RCT-02)"))
    if lim:
        for a in lim["ambiguities"]:
            manual.append(("ambiguity", a["ref"], a["msg"]))
        for l in lim["lines"]:
            if l["deducted"] > 0:
                refs.add(l["ref"])
                unit = {"POL-PD-01": "meals", "POL-PD-02": "lodging", "POL-PD-03": "ground transport"}[l["ref"]]
                notes.append(f"{unit} of {money(l['claimed'])} exceeds the cap of {money(l['cap'])} ({l['ref']}), {money(l['deducted'])} deducted")
    if tim and tim["late"]:
        manual.append(("policy", "POL-TIME-01", f"submitted {tim['days_since_expense']} days after the expense (POL-TIME-01 allows 30)"))
    if tier:
        if tier["needs_review"] or not manual:
            refs.add(tier["ref"])
        if tier["needs_review"]:
            manual.append(("policy", "POL-APR-03", f"reimbursable total {money(tier['total'])} is above the $2,000 agent limit (POL-APR-03), director approval required"))

    for kind, ref, _ in manual:
        if ref:
            refs.add(ref)

    reimbursable = lim["reimbursable_total"] if lim else 0.0
    cap_ded = lim["cap_deductions"] if lim else 0.0

    if manual:
        decision, approved, deducted = "MANUAL_REVIEW", 0.0, 0.0
        confidence = 0.70 if any(k == "ambiguity" for k, _, _ in manual) else 0.90
        explanation = ("Routed to manual review: " + "; ".join(m for _, _, m in manual) + ". "
                       f"Nothing is finalised; if the reviewer clears it, the reimbursable amount would be {money(reimbursable)}"
                       + (f" with {money(ineligible_total + cap_ded)} deducted." if ineligible_total + cap_ded else "."))
    elif elig["eligible_count"] == 0:
        decision, approved, deducted, confidence = "REJECT", 0.0, ineligible_total, 0.97
        explanation = "; ".join(notes) + f". Nothing in this claim is reimbursable, so the full {money(ineligible_total)} is deducted."
    else:
        approved, deducted = reimbursable, round(ineligible_total + cap_ded, 2)
        if deducted > 0:
            decision, confidence = "PARTIAL_APPROVE", 0.93
            explanation = ("; ".join(notes)[0].upper() + "; ".join(notes)[1:] + f". Everything else is compliant. Reimbursing {money(approved)} "
                           f"({tier['tier']} tier, {tier['ref']}) and deducting {money(deducted)}.")
        else:
            decision, confidence = "APPROVE", 0.96
            explanation = (f"All {elig['eligible_count']} items are eligible (POL-CAT-01), required receipts are attached and amounts are within "
                           f"the per-diem/nightly caps. Total {money(approved)} falls in the {tier['tier']} tier ({tier['ref']}) and can be approved.")

    return {"decision": decision, "approved_amount": round(approved, 2), "deducted_amount": round(deducted, 2),
            "missing_docs": missing_docs, "policy_refs": sorted(refs), "confidence": confidence,
            "explanation": explanation, "claimed": claimed}

In [ ]:
SYSTEM_PROMPT = """You review employee travel reimbursement claims against a company policy.
Work like a careful reviewer:
- Start with lookup_policy to ground yourself in the relevant rules.
- Then use the check_* tools as needed. Skip checks that cannot affect the outcome (e.g. nothing on the claim is eligible).
- Do not guess. If something is missing, ambiguous, in conflict, or a policy exception, the answer is MANUAL_REVIEW.
- When you have enough, call submit_decision with the decision, a confidence between 0 and 1,
  and a 2-3 sentence explanation that cites POL-* ids. Do not invent amounts; the tools compute them."""

_claim_id_schema = {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_policy",
            "description": "Search the travel policy for rules relevant to a query.",
            "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_eligibility",
            "description": "Classify each line item as eligible/ineligible/unknown and flag airfare-class exceptions and total mismatches.",
            "parameters": _claim_id_schema
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_receipts",
            "description": "Find eligible items that need a receipt but don't have one.",
            "parameters": _claim_id_schema
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_limits",
            "description": "Apply per-diem / nightly / daily caps and compute the reimbursable total.",
            "parameters": _claim_id_schema
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_timeliness",
            "description": "Check the 30-day submission window.",
            "parameters": _claim_id_schema
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_approval_tier",
            "description": "Find the approval tier for a reimbursable total (use the total from check_limits).",
            "parameters": {"type": "object", "properties": {"reimbursable_total": {"type": "number"}}, "required": ["reimbursable_total"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_decision",
            "description": "Submit the final decision once the checks are done.",
            "parameters": {"type": "object", "properties": {
                "decision": {"type": "string", "enum": DECISIONS},
                "confidence": {"type": "number"},
                "explanation": {"type": "string"}}, "required": ["decision", "confidence", "explanation"]}
        }
    }
]


def llm_explore(cid, findings, max_turns=8):
    """Let the model pick tools using OpenAI function calling. Returns its submitted decision (or None)."""
    c = CLAIMS[cid]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Review this claim and submit a decision.\n\n" + json.dumps(c, indent=2)}
    ]
    submitted = None
    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        msg = response.choices[0].message
        messages.append(msg)
        
        if not msg.tool_calls:
            break
            
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            try:
                args = json.loads(tool_call.function.arguments)
            except Exception:
                args = {}
                
            if name == "submit_decision":
                submitted = args
                out = {"ok": True}
            elif name in TOOL_FUNCS:
                out = call_tool(cid, name, args, findings, "llm")
            else:
                out = {"error": f"unknown tool {name}"}
                
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(out)
            })
        if submitted:
            break
    return submitted


def run_claim(cid):
    AUDIT[cid] = []
    findings, llm_out = {}, None

    if MODE == "llm":
        try:
            llm_out = llm_explore(cid, findings)
        except Exception as e:   
            AUDIT[cid].append({"caller": "harness", "tool": "llm_error", "args": {}, "output": {"error": str(e)}})
            findings.clear(); AUDIT[cid].clear()

    complete_findings(cid, findings)
    d = decide(cid, findings)

    if llm_out:
        if llm_out["decision"] == d["decision"]:
            d["explanation"] = llm_out["explanation"].strip()
        else:
            AUDIT[cid].append({"caller": "harness", "tool": "llm_disagreement", "args": {},
                               "output": {"llm": llm_out["decision"], "rules": d["decision"], "used": "rules"}})

    tools_used = []
    for step in AUDIT[cid]:
        if step["tool"] in TOOL_FUNCS and step["tool"] not in tools_used:
            tools_used.append(step["tool"])
    tools_used.append("validate_output")

    result = {"claim_id": cid, "decision": d["decision"], "approved_amount": d["approved_amount"],
              "deducted_amount": d["deducted_amount"], "missing_docs": d["missing_docs"],
              "policy_refs": d["policy_refs"], "confidence": d["confidence"],
              "explanation": d["explanation"], "tools_used": tools_used}

    check = validate_output(result, d["claimed"])
    AUDIT[cid].append({"caller": "harness", "tool": "validate_output", "args": {}, "output": check})
    if not check["valid"]:   
        result.update({"decision": "MANUAL_REVIEW", "approved_amount": 0.0, "deducted_amount": 0.0, "confidence": 0.5,
                       "explanation": "Automatic result failed validation (" + "; ".join(check["errors"]) + "), sent to manual review."})
    return result

## 5. Run all claims

In [ ]:
results = [run_claim(c["claim_id"]) for c in CLAIMS_RAW]

summary = pd.DataFrame(results)[["claim_id", "decision", "approved_amount", "deducted_amount", "confidence", "tools_used"]]
summary["tools_used"] = summary["tools_used"].apply(lambda t: ", ".join(t))
summary

### Sample outputs with explanations

All five claims, with the explanation the agent produced for each:

In [ ]:
for r in results:
    print(f"{r['claim_id']}  ->  {r['decision']}   approved {money(r['approved_amount'])} | deducted {money(r['deducted_amount'])} | confidence {r['confidence']}")
    print(f"   refs: {', '.join(r['policy_refs'])}")
    if r["missing_docs"]:
        print(f"   missing: {'; '.join(r['missing_docs'])}")
    print(f"   {r['explanation']}\n")

## 6. Audit trail

Every tool call is logged with who made it (`llm` or `harness`), the arguments and the output. The cell below prints the trail for the hardest claim (CLM-004: business-class fare, missing receipt, over $2,000).

In [ ]:
def show_audit(cid):
    print(f"=== {cid} ===")
    for n, step in enumerate(AUDIT[cid], 1):
        out = json.dumps(step["output"], default=str)
        out = out if len(out) < 260 else out[:257] + "..."
        print(f"{n}. [{step['caller']}] {step['tool']}({json.dumps(step['args']) if step['args'] else ''})\n     -> {out}")

show_audit("CLM-004")

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    picker = widgets.Dropdown(options=list(CLAIMS), description="Claim:")
    panel = widgets.Output()
    def _on_change(change):
        with panel:
            clear_output()
            show_audit(change["new"])
    picker.observe(_on_change, names="value")
    display(picker, panel)
except ImportError:
    print("\n(ipywidgets not installed - skipping the claim picker)")

## Dashboard

Built from the `results` list above. Left: how many claims landed in each decision. Right: for each claim, how the claimed amount splits into approved, deducted and *held for review* (claimed but not yet decided).

In [ ]:
df = pd.DataFrame(results)
df["claimed"] = df["claim_id"].map(lambda cid: sum(i["amount"] for i in CLAIMS[cid]["items"]))
df["held"] = (df["claimed"] - df["approved_amount"] - df["deducted_amount"]).round(2)

colors = {"APPROVE": "#2e7d32", "PARTIAL_APPROVE": "#f9a825", "REJECT": "#c62828", "MANUAL_REVIEW": "#1565c0"}
counts = df["decision"].value_counts().reindex(DECISIONS, fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.5]})

bars = ax1.bar(counts.index, counts.values, color=[colors[d] for d in counts.index])
ax1.set_title("Decisions")
ax1.set_ylabel("Claims")
ax1.set_yticks(range(0, int(counts.max()) + 2))
ax1.tick_params(axis="x", rotation=20)
for b, v in zip(bars, counts.values):
    ax1.text(b.get_x() + b.get_width() / 2, v + 0.05, str(v), ha="center")

ax2.bar(df["claim_id"], df["approved_amount"], label="Approved", color="#2e7d32")
ax2.bar(df["claim_id"], df["deducted_amount"], bottom=df["approved_amount"], label="Deducted", color="#c62828")
ax2.bar(df["claim_id"], df["held"], bottom=df["approved_amount"] + df["deducted_amount"], label="Held for review", color="#90a4ae")
ax2.set_title("Claimed amount by outcome (USD)")
ax2.set_ylabel("USD")
ax2.legend()
for i, r in df.iterrows():
    ax2.text(i, r["claimed"] + 40, f"{r['decision'].replace('_', ' ').title()}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("UI SS_1.png", dpi=150)
plt.show()

totals = pd.DataFrame({"USD": [df["claimed"].sum(), df["approved_amount"].sum(), df["deducted_amount"].sum(), df["held"].sum()]},
                      index=["Claimed", "Approved", "Deducted", "Held for review"])
totals

In [ ]:
print(json.dumps(results, indent=2))